# Goal

5-й шаг `42`-линейки (продолжение `42.4` → `42.5b`/`42.5c`). База — лучший агент `:160` из `42.4`.
Шатаем ОДНУ ось: **форму аннилинга** `learn_rate` (`ent_coef` держим const, кроме явной проверки).

Гипотеза: политика «кристаллизуется» лучше, если температуру (lr) ПЛАВНО отпускать
(warmup → спад), а не держать `const`.

КОНФАУНД, который разводим: низкий `const` может выигрывать просто потому, что меньше
ломает warm-start-родителя — т.е. это про УРОВЕНЬ lr, а не про форму кривой.
Поэтому контроль подобран LEVEL-MATCHED: у `const_match`/`linear_down`/`shark_fin`
одинаковое СРЕДНЕЕ по времени lr ≈ 5.5e-5, различается только ФОРМА.
Если формы не отличаются от `const_match`, а `const_lo` (ниже) лучше — значит правит уровень, не форма.

Армсы (`schedule`; всё прочее идентично, включая ent `const(0.023)`):
- `const_lo`      — `const(0.00003)`                 ; mean=3.00e-5  (контроль «низкий ровный», фаворит 42.5c)
- `const_match`   — `const(0.000055)`                ; mean=5.50e-5  (LEVEL-MATCHED контроль, форма=плоская)
- `linear_down`   — `linear(0.0001, 0.00001)`        ; mean=5.50e-5  (matched, форма=прямой спад)
- `shark_fin`     — `shark_fin(0.0001, 0.00001, 0.05)` ; mean=5.47e-5 (matched, форма=warmup→косинус)
- `shark_fin_hot` — `shark_fin(0.00025, 0.00001, 0.05)` ; mean=1.30e-4 (горячий старт / таск-сдвиг)

Повторности: `random_seed` тоже сэмплится из сетки → 11 сидов × 5 арм = 55 ячеек,
`launches_count=50` → ровно 10 прогонов на арм (сбалансировано). Парная статистика по общим сидам.

# TARGET_NOTEBOOK_FNAME


In [1]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_07_old.ipynb'

# OPTUNA_DECISIVE_METRIC


In [2]:
OPTUNA_DECISIVE_METRIC = 'test/levels_passed_mean'

# GRID_SEARCH_SPACE


In [3]:
# @launchit.collect
GRID_SEARCH_SPACE = dict(
    schedule=[
        'const_lo',
        'const_match',
        'linear_down',
        'shark_fin',
        'shark_fin_hot',
    ],
    random_seed=list(range(1, 111, 10)),  # 11 seeds -> grid 5x11=55, launches_count=50 => 10 per arm
)

# set_hyperparameters


In [4]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    HP.general.random_seed = optuna_trial.suggest_categorical('random_seed', GRID_SEARCH_SPACE['random_seed'])
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True

    HP.env.count = 32
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    wm_parent = '18d_world_model_09:183'
    HP.vision_head.parent = dict(model=wm_parent, weights=wm_parent)
    HP.vision_head.is_trainable = False

    HP.encoder.parent = dict(model=wm_parent, weights=wm_parent)
    HP.encoder.is_trainable = True

    HP.agent.parent = None
    HP.agent.sequence_length = 4
    HP.agent.action_plan_length = 10
    HP.agent.d_model = 256
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True

    parent = '18n_ppo_tr_frostbite_07_old:160'
    HP.encoder.parent['weights'] = parent
    HP.agent.parent = parent

    HP.test.env_rams = None
    HP.test.env_ram_patches = None
    HP.test.break_on_level_passed = False

    # Rollout: keep nine_lives (deep chains) + natural play.
    HP.ppo.rollout_env_rams = None
    HP.ppo.rollout_env_ram_patches = [
        ['nine_lives'],
    ]

    # Training procedure params (PPO related)
    HP.ppo.global_steps_count = 3_000_000
    HP.ppo.rollout_steps_count = 512
    HP.ppo.epochs_count = 2
    HP.ppo.batch_size = 512
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.vf_coef = 0.2
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1

    # ---- THE ONE AXIS UNDER TEST: annealing SHAPE of learn_rate ----
    # All arms share every other HP. ent_coef stays const(0.023) so only lr's SHAPE moves.
    # Three arms are LEVEL-MATCHED (time-mean lr ~5.5e-5) and differ ONLY in shape,
    # which is what separates "shape matters" from "lower level matters".
    schedule = optuna_trial.suggest_categorical('schedule', GRID_SEARCH_SPACE['schedule'])
    if schedule == 'const_lo':
        # low flat control (42.5c favourite); NOT level-matched -> captures the "level" effect
        HP.ppo.learn_rate = 'const(0.00003)'
    elif schedule == 'const_match':
        # flat control, level-matched to the annealed arms (time-mean ~5.5e-5)
        HP.ppo.learn_rate = 'const(0.000055)'
    elif schedule == 'linear_down':
        # level-matched, straight decay shape
        HP.ppo.learn_rate = 'linear(0.0001, 0.00001)'
    elif schedule == 'shark_fin':
        # level-matched, warmup -> cosine decay shape (the "cool the pan" arm)
        HP.ppo.learn_rate = 'shark_fin(0.0001, 0.00001, 0.05)'
    elif schedule == 'shark_fin_hot':
        # same shape from a hotter peak (cold-Adam-shock / task-shift arm)
        HP.ppo.learn_rate = 'shark_fin(0.00025, 0.00001, 0.05)'
    else:
        assert False, schedule
    HP.ppo.ent_coef = 'const(0.023)'

    HP.ppo.gamma = 0.997
    HP.ppo.gae_lambda = 0.95
    HP.ppo.clip_coef = 0.1
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5
    HP.ppo.target_kl = None
    HP.ppo.norm_adv = True

    return HP

# Results

Проверяем ОДНУ ось — **форму расписания `HP.ppo.learn_rate` внутри шага** (warmup→спад против ровного `const`).

Всё прочее идентично `42.5c`: база `18n_ppo_tr_frostbite_07_old` + `VisionHead:183` (frozen), parent ЖЁСТКО зашит `07_old:160`, `global_steps_count=3M`, rollout `[nine_lives]`, cold-тест (30 env, `break_on_level_passed=False`), `clip_coef=0.12`, `ent_coef=const(0.023)`. `random_seed` сэмплится из сетки (11 сидов) → **10 прогонов на руку**, парный дизайн по сидам (руки делят сиды).

Руки различаются ТОЛЬКО формой `learn_rate`:

| рука | расписание | средний lr по времени | что проверяет |
|---|---|---|---|
| `const_lo` | `const(0.00003)` | 3.00e-5 | низкий ровный (не matched) — эффект УРОВНЯ |
| `const_match` | `const(0.000055)` | 5.50e-5 | контроль, форма = плоская |
| `linear_down` | `linear(0.0001, 0.00001)` | 5.50e-5 | matched, форма = прямой спад |
| `shark_fin` | `shark_fin(0.0001, 0.00001, 0.05)` | 5.47e-5 | matched, форма = warmup→косинус |
| `shark_fin_hot` | `shark_fin(0.00025, 0.00001, 0.05)` | 1.30e-4 | горячий старт (шок/таск-сдвиг) |

`const_match` / `linear_down` / `shark_fin` — **level-matched** (средний по времени lr ≈5.5e-5), различаются только ФОРМОЙ. Только так «форма кривой» отделяется от «уровня lr».

**Применение расписаний подтверждено:** TB `anneal/a) learn_rate` по всем 50 прогонам — **0/50 несоответствий** спекам (`const_*` плоские, `linear_down` убывает 1e-4→1e-5, `shark_fin`/`_hot` стартуют с 0). То есть null — не «ручку не применили».

| метрика | `18n_study_42.5d` | `18n_study_42.5c` | `18n_study_25.5` |
|---|---|---|---|
| Sample Size (n) | 50 | 50 | 15 |
| Mean Score | **4.0500** | 3.9206 | 4.0250 |
| Best Value | **4.5000** (`:245`, `:256`) | 4.4375 (`:200`) | 4.7500 (`:247`) |
| Standard Deviation | **0.1969** | 0.4394 | 0.3000 |
| Shapiro-Wilk (p) | 0.482 | 5.3e-07 | 0.023 |
| Welch's t-test (p) | — | 0.062 | 0.765 |
| Mann-Whitney U (p) | — | 0.396 | 0.198 |
| Cohen's d | — | +0.38 | +0.11 |
| Statistical Result | — | н/з | н/з |

Разбивка по рукам (n=10 на руку):

| рука | n | Mean | Best (модель) | Std | Shapiro (p) | Welch vs `const_match` (p) | MWU vs `const_match` (p) | Cohen's d |
|---|---|---|---|---|---|---|---|---|
| `const_lo` | 10 | 4.0281 | 4.2188 (`:266`) | 0.1288 | 0.961 | 0.855 | 0.909 | +0.08 |
| `const_match` | 10 | 4.0125 | 4.3125 (`:268`) | 0.2321 | 0.309 | — | — | — |
| `linear_down` | 10 | 4.1156 | 4.5000 (`:245`) | 0.1869 | 0.640 | 0.289 | 0.543 | +0.49 |
| `shark_fin` | 10 | 4.0219 | 4.2500 (`:254`) | 0.1517 | 0.792 | 0.916 | 0.939 | +0.05 |
| `shark_fin_hot` | 10 | 4.0719 | 4.5000 (`:256`) | 0.2728 | 0.612 | 0.607 | 0.495 | +0.23 |

Парно по сидам (дельта = рука − `const_match`):

| контраст | n пар | Δ (рука − `const_match`) | paired t (p) | Wilcoxon (p) |
|---|---|---|---|---|
| `linear_down` − `const_match` | 9 | +0.0521 | 0.534 | 0.633 |
| `shark_fin_hot` − `const_match` | 9 | +0.0729 | 0.624 | 0.844 |
| `shark_fin` − `const_match` | 9 | +0.0417 | 0.683 | 0.547 |
| `const_lo` − `const_match` | 9 | +0.0139 | 0.868 | 0.801 |

Что видно и чего не видно:
- **ФОРМА кривой не отличима от плоской** при равном среднем lr: все `Δ(рука − const_match)` в диапазоне +0.014…+0.073, ни один не значим (p ≥ 0.29), 95% CI всех дельт накрывают 0. ANOVA `test ~ arm`: **F=0.46, p=0.76**, η²=**3.9%** — руки объясняют 4% дисперсии.
- **УРОВЕНЬ lr тоже не отличим** в узкой зоне: `const(3e-5)` vs `const(5.5e-5)` — Δ=+0.014, p=0.87.
- **Тест имел мощность:** MDE (парный, n=9, 80%, α=0.05) ≈ **0.32 уровня**; наблюдаемый разброс средних между руками = **0.103**. Эффект ≥0.32 мы бы увидели ⇒ null осмысленный, не «слепой тест».
- **Бюджет шума:** между руками sd = 0.039, внутри руки (сид/шум) sd = **0.201** — эффект руки в 5 раз меньше шума.
- **Метрика у потолка:** `rollout episode_lp_max` у ВСЕХ рук ≈6.0; полный размах рук 0.10 при sd 0.20 — измеритель не различает.

Системная проблема: **не проверено по журналу** (журнал несёт только решающий метрик; роллаут-хвост `episode_lp` 4.46–4.68 без значимых различий).

<img src="./img/42_5d_summary.png">

**Выводы**
1) **Форма `learn_rate` внутри шага НЕ работает** на этой метрике: warmup→косинус (`shark_fin`), прямой спад (`linear_down`) и горячий старт (`shark_fin_hot`) — все статистически неотличимы от плоского `const` при равном среднем lr. Гипотеза «плавный аннилинг кристаллизует политику» на шаге 5 **не подтверждена**.
2) **И уровень lr в узкой зоне не работает:** `const(3e-5)` vs `const(5.5e-5)` — н/з. Прошлый вывод из `42.5c` («правит низкий lr») верен для широкого диапазона (2.5e-5…2.5e-4), но внутри низа всё плоско.
3) **Шаг 5 уже заморожен.** Пластичность (сдвиг `episode_lp` внутри прогона) по шагам: step1 +647% → step3 +51% → step4 +12% → **step5 +1%**. Расписание нечему проявить: агент приходит сформированным и почти не двигается. Отсюда null.
4) Подъём относительно `42.5c` (mean 3.92 → 4.05, sd 0.44 → 0.20) дал НЕ аннилинг, а hardcoded parent `:160` + фиксированный `clip_coef=0.12`.
5) **Следующий шаг:** пробовать аннилинг на **пластичном** шаге — **шаг 3** (последний живой, +51% сдвига, запас до потолка 2.45 vs 5.28). Там parent-выбор в `42.3` сдвигал метрику на 0.37 (2.28…2.65), т.е. шаг чувствителен к ручкам. Дизайн: фиксированный parent (не сэмплить), те же level-matched руки, **n≥15-20/руку** (шум шага 3 выше: within-parent sd 0.478 против 0.302 у шага 5 ⇒ при n=10 MDE ≈0.48).

# System

In [5]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [6]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07_old.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_07_old',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_42.5d/18n_study_42.5d.ipynb',
 'optuna_study_name': '18n_study_42.5d',
 'optuna_study_serial': '42.5d',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_42.5d/18n_study_42.5d.optuna'}

In [7]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [8]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [9]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC=OPTUNA_DECISIVE_METRIC,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [10]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [11]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 50
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-15 10:08:30,579] A new study created in Journal with name: 18n_study_42.5d


2026.10.15-10:08:31.018553     0.415 >> Model instance registered, version=232
2026.10.15-10:08:31.030437     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch232.ipynb"
2026.10.15-10:08:31.031705     0.001 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07_old:232"; running launches=1
2026.10.15-10:09:02.034350     0.308 >> Model instance registered, version=233
2026.10.15-10:09:02.049575     0.007 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch233.ipynb"
2026.10.15-10:09:02.050056     0.008 >> 5.5 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07_old:233"; running launches=2
2026.10.15-10:09:33.081365     0.341 >> Model instance registered, version=234
2026.10.15-10:09:33.093055     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch234.ipynb"
2026.10.15-10:09:33.093594     0.005 >> 4.8 idle runners exist, submitted launch "18n

In [12]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-10-15 13:43:56,932] Using an existing study with name '18n_study_42.5d' instead of creating a new one.


2026.10.15-13:43:56.933614     5.033 >> Study statistics: 
2026.10.15-13:43:56.937822     0.004 >> 	Number of finished trials: 50
2026.10.15-13:43:56.938631     0.001 >> 	Number of pruned trials: 0
2026.10.15-13:43:56.939255     0.001 >> 	Number of complete trials: 50
2026.10.15-13:43:56.939840     0.001 >> Best trial:
2026.10.15-13:43:56.940817     0.001 >> 	Value: 4.5
2026.10.15-13:43:56.941191     0.000 >> 	Model version: 245
2026.10.15-13:43:56.941557     0.000 >> 	Params: 
2026.10.15-13:43:56.941923     0.000 >> 		random_seed: 41
2026.10.15-13:43:56.942260     0.000 >> 		schedule: linear_down


In [13]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.10.15-13:43:56.951163     0.009 >> Top trials:
2026.10.15-13:43:56.951936     0.001 >> 1) 18n_ppo_tr_frostbite_07_old:245, value=4.5
2026.10.15-13:43:56.952506     0.001 >> 2) 18n_ppo_tr_frostbite_07_old:256, value=4.5
2026.10.15-13:43:56.952829     0.000 >> 3) 18n_ppo_tr_frostbite_07_old:268, value=4.3125


In [14]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(4.05), np.float64(0.19495592450602778))

In [15]:
f'{3.207657610218081e-05:.6f}'

'0.000032'